LangChain 的 create_agent 在底层运行于 LangGraph 的运行时之上。
LangGraph 公开了一个包含以下信息的 Runtime 对象：
1. Context：静态信息，例如用户 ID、数据库连接，或智能体调用所需的其他依赖项
2. Store：用于长期记忆的 BaseStore 实例
3. Stream writer：用于通过 "custom" 流模式流式传输信息的对象
4. Execution info：当前执行的标识和重试信息（线程 ID、运行 ID、尝试次数）
5. Server info：在 LangGraph Server 上运行时的特定于服务器的元数据（助手 ID、图 ID、已认证用户）

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass 
class Context:
    user_name: str

agent=create_agent(
    model="deepseek:deepseek-v4-pro",
    context_schema=Context
    )

agent.invoke(
    {
        "messages":[
        {
                "role":"system",
                "content":"你叫什么名字",
                
        }
    ]
    },context=Context(user_name="张三")
)

在工具内部访问上下文

In [ ]:
from dataclasses import dataclass

from langchain.tools import ToolRuntime, tool


@dataclass
class Context:
    user_id: str

@tool
def fetch_user_email_preferences(runtime: ToolRuntime[Context]) -> str:
    """Fetch the user's email preferences from the store."""
    user_id = runtime.context.user_id  

    preferences: str = "The user prefers you to write a brief and polite email."
    if runtime.store:
        if memory := runtime.store.get(("users",), user_id):
            preferences = memory.value["preferences"]

    return preferences

在工具内部的执行信息和服务器信息


In [ ]:
from langchain.tools import ToolRuntime, tool


@tool
def context_aware_tool(runtime: ToolRuntime) -> str:
    """A tool that uses execution and server info."""
    # Access thread and run IDs
    info = runtime.execution_info
    print(f"Thread: {info.thread_id}, Run: {info.run_id}")

    # Access server info (only available on LangGraph Server)
    server = runtime.server_info
    if server is not None:
        print(f"Assistant: {server.assistant_id}")
        if server.user is not None:
            print(f"User: {server.user.identity}")

    return "done"

在中间件内部访问运行时上下文

In [ ]:
from dataclasses import dataclass

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    ModelRequest,
    after_model,
    before_model,
    dynamic_prompt,
)
from langchain.messages import AnyMessage
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_name: str

# Dynamic prompts
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name  
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt

# Before model hook
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Processing request for user: {runtime.context.user_name}")
    return None

# After model hook
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Completed request for user: {runtime.context.user_name}")
    return None

agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    middleware=[dynamic_system_prompt, log_before_model, log_after_model],
    context_schema=Context
)

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)